In [4]:
import torchvision
import os, shutil, random
from PIL import Image
import numpy as np

In [ ]:
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

for split, train in [('train', True), ('test', False)]:
    dataset = torchvision.datasets.CIFAR10(root='./CIFAR-10', train=train, download=True)
    for cls in classes:
        os.makedirs(f'./cifar-10/{split}/{cls}', exist_ok=True)
    for i, (img, label) in enumerate(dataset):
        img.save(f'./cifar-10/{split}/{classes[label]}/{i}.png')

100%|██████████| 170498071/170498071 [00:15<00:00, 11066695.62it/s]


Extracting ./CIFAR-10/cifar-10-python.tar.gz to ./CIFAR-10
Files already downloaded and verified


In [2]:
random.seed(42)

src = "train"
val_dst = "val"
val_split = 0.1  # 10% → 5,000 images for val

# Assumes subfolders per class: train/airplane/, train/car/, etc.
for class_name in os.listdir(src):
    class_dir = os.path.join(src, class_name)
    if not os.path.isdir(class_dir):
        continue

    images = os.listdir(class_dir)
    random.shuffle(images)
    n_val = int(len(images) * val_split)

    val_images = images[:n_val]

    for img in val_images:
        dst_dir = os.path.join(val_dst, class_name)
        os.makedirs(dst_dir, exist_ok=True)
        shutil.move(os.path.join(class_dir, img), os.path.join(dst_dir, img))

print("Done! Val split created.")

Done! Val split created.


In [6]:
# CIFAR-10 class → integer label mapping
CLASS_TO_IDX = {
    "airplane": 0, "automobile": 1, "bird": 2, "cat": 3, "deer": 4,
    "dog": 5, "frog": 6, "horse": 7, "ship": 8, "truck": 9
}

def save_split(split_dir, split_name):
    images, labels = [], []

    for class_name, label in CLASS_TO_IDX.items():
        class_dir = os.path.join(split_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in sorted(os.listdir(class_dir)):  # sorted for reproducibility
            if not fname.endswith(".png"):
                continue
            img = Image.open(os.path.join(class_dir, fname)).convert("RGB")
            img_array = np.array(img)          # [H, W, C]
            img_array = img_array.transpose(2, 0, 1)  # → [C, H, W]
            images.append(img_array)
            labels.append(label)

    x = np.stack(images)          # [N, C, H, W]
    y = np.array(labels)          # [N]

    np.save(f"{split_name}_x.npy", x)
    np.save(f"{split_name}_y.npy", y)
    print(f"{split_name}: x={x.shape}, y={y.shape}")

save_split("train", "train")
save_split("val",   "valid")
save_split("test",  "test")

train: x=(45000, 3, 32, 32), y=(45000,)
valid: x=(5000, 3, 32, 32), y=(5000,)
test: x=(10000, 3, 32, 32), y=(10000,)
